# Milestone 4

Saebastion Cole

11/14/2026

DCS540-T301 Data Preperation

In [64]:
# Imports
import re
import requests
import pandas as pd

# These imports support retrying slow/failed requests; otherwise a single timeout kills the whole download.
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [65]:
# This is the base URL of the API I will be using
BASE_URL = "https://api.worldbank.org/v2"
TIMEOUT = 120  # the World Bank API can be slow for large pages

# A session that automatically retries on timeouts / 5xx errors
session = requests.Session()

# This will handle retry logic if it fails for some of the common HTTP errors
retry = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry))

## Get the GDP Data

In [66]:
# This is the URL to get the GDP data
url = f"{BASE_URL}/country/all/indicator/NY.GDP.PCAP.CD"
params = {"format": "json", "date": "2010:2026", "per_page": 20000}

response = session.get(url, params=params, timeout=TIMEOUT)
response.raise_for_status()

metadata, records = response.json()
print(metadata)

# make a quick dataframe to show we have the data
gdp = pd.DataFrame({
    "iso3": [r["countryiso3code"] for r in records],
    "year": [int(r["date"]) for r in records],
    "gdp_per_capita_usd": [r["value"] for r in records],
})

{'page': 1, 'pages': 1, 'per_page': 20000, 'total': 4256, 'sourceid': '2', 'lastupdated': '2026-11-11'}


In [67]:
# This will show the simplified data
print(len(gdp), "rows")
gdp.head()

4256 rows


,iso3,year,gdp_per_capita_usd
0,AFE,2025,1722.385620
1,AFE,2024,1628.227289
2,AFE,2023,1571.132704
3,AFE,2022,1675.902524
4,AFE,2021,1560.894626


## Get the Country Metadata

This will set the countries to all to get everything the API has available

In [68]:
r = session.get(f"{BASE_URL}/country", params={"format": "json", "per_page": 400}, timeout=TIMEOUT)
r.raise_for_status()
_, countries_raw = r.json()

countries = pd.DataFrame({
    "iso3": [c["id"] for c in countries_raw],
    "iso2": [c["iso2Code"] for c in countries_raw],
    "name": [c["name"] for c in countries_raw],
    "region": [c["region"]["value"] for c in countries_raw],
    "income_level": [c["incomeLevel"]["value"] for c in countries_raw],
    "capital": [c["capitalCity"] for c in countries_raw],
    "longitude": pd.to_numeric([c["longitude"] for c in countries_raw], errors="coerce"),
    "latitude": pd.to_numeric([c["latitude"] for c in countries_raw], errors="coerce"),
})
countries = countries[countries["region"] != "Aggregates"].reset_index(drop=True)
print(len(countries), "countries")
countries.head()

217 countries


,iso3,iso2,name,region,income_level,capital,longitude,latitude
0,ABW,AW,Aruba,Latin America & Caribbean,High income,Oranjestad,-70.0167,12.51670
1,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.52280
2,AGO,AO,Angola,Sub-Saharan Africa,Lower middle income,Luanda,13.2420,-8.81155
3,ALB,AL,Albania,Europe & Central Asia,Upper middle income,Tirane,19.8172,41.33170
4,AND,AD,Andorra,Europe & Central Asia,High income,Andorra la Vella,1.5218,42.50750


## Merge GDP with the Country Metadata

Joining the two tables together to have the GDP data with the country data

In [69]:
countries_merged = countries.merge(gdp, on="iso3", how="inner")
countries_merged = countries_merged.sort_values(["name", "year"]).reset_index(drop=True)
print(countries_merged.shape)
countries_merged.head()

(3472, 10)


,iso3,iso2,name,region,income_level,capital,longitude,latitude,year,gdp_per_capita_usd
0,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.5228,2010,560.621505
1,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.5228,2011,606.694676
2,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.5228,2012,651.417134
3,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.5228,2013,637.087099
4,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.5228,2014,625.054942


In [70]:
countries_merged.info()
countries_merged.isna().mean().round(3)

<class 'pandas.DataFrame'>
RangeIndex: 3472 entries, 0 to 3471
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   iso3                3472 non-null   str    
 1   iso2                3472 non-null   str    
 2   name                3472 non-null   str    
 3   region              3472 non-null   str    
 4   income_level        3472 non-null   str    
 5   capital             3472 non-null   str    
 6   longitude           3376 non-null   float64
 7   latitude            3376 non-null   float64
 8   year                3472 non-null   int64  
 9   gdp_per_capita_usd  3339 non-null   float64
dtypes: float64(3), int64(1), str(6)
memory usage: 271.4 KB


iso3                  0.000
iso2                  0.000
name                  0.000
region                0.000
income_level          0.000
capital               0.000
longitude             0.028
latitude              0.028
year                  0.000
gdp_per_capita_usd    0.038
dtype: float64

## Transformation 1: Normalize the headers
I will rename the columns to match the format I used in Milestones 2 and 3. All headers become kebab casing, and the ISO code columns use the same names as the ISO-3166 dataset so this data lines up with that file when I combine the sources later.

In [71]:
# First data manipulation: rename the ISO code columns to match the ISO-3166 dataset from Milestone 2,
# then convert all headers to kebab casing (lowercase, hyphens instead of underscores/spaces)
countries_merged = countries_merged.rename(columns={'iso3': 'alpha-3', 'iso2': 'alpha-2'})
countries_merged.columns = countries_merged.columns.str.strip().str.lower().str.replace('_', '-').str.replace(' ', '-')

# Put the identifying columns first, the same way I ordered them in Milestone 2
countries_merged = countries_merged[['name', 'alpha-2', 'alpha-3', 'region', 'income-level', 'capital',
         'longitude', 'latitude', 'year', 'gdp-per-capita-usd']]
countries_merged.head()

,name,alpha-2,alpha-3,region,income-level,capital,longitude,latitude,year,gdp-per-capita-usd
0,Afghanistan,AF,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.5228,2010,560.621505
1,Afghanistan,AF,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.5228,2011,606.694676
2,Afghanistan,AF,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.5228,2012,651.417134
3,Afghanistan,AF,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.5228,2013,637.087099
4,Afghanistan,AF,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Kabul,69.1761,34.5228,2014,625.054942


## Transformation 2: Index by country name and year
Each row is one country in one year, so I will make name and year the index. This makes the data easier to read and lets me look up a country (or a country in a given year) directly. Before setting the index I check that each name/year pair only appears once, since an index should not have duplicates.

In [72]:
# Set the country name and year as the index
# Check there are no duplicate name/year pairs first
print(countries_merged.duplicated(subset=['name', 'year']).sum(), "duplicate name/year rows")

countries_merged = countries_merged.set_index(['name', 'year']).sort_index()
countries_merged.head()

0 duplicate name/year rows


alpha-2 alpha-3  \
name        year                   
Afghanistan 2010      AF     AFG   
            2011      AF     AFG   
            2012      AF     AFG   
            2013      AF     AFG   
            2014      AF     AFG   

                                                             region  \
name        year                                                      
Afghanistan 2010  Middle East, North Africa, Afghanistan & Pakistan   
            2011  Middle East, North Africa, Afghanistan & Pakistan   
            2012  Middle East, North Africa, Afghanistan & Pakistan   
            2013  Middle East, North Africa, Afghanistan & Pakistan   
            2014  Middle East, North Africa, Afghanistan & Pakistan   

                 income-level capital  longitude  latitude  gdp-per-capita-usd  
name        year                                                                
Afghanistan 2010   Low income   Kabul    69.1761   34.5228          560.621505  
            2011   Low income   Kabul    69.1761   34.5228          606.694676  
            2012   Low income   Kabul    69.1761   34.5228          651.417134  
            2013   Low income   Kabul    69.1761   34.5228          637.087099  
            2014   Low income   Kabul    69.1761   34.5228          625.054942

## Transformation 3: Identify outliers and bad data
Looking at the data, some countries have a blank string for their capital instead of a missing value, so they would not show up as missing. I will convert those to proper missing values. A few countries have no GDP value for any year, so I will remove them since they add nothing to a GDP dataset. Countries missing only some years stay in, since leaving those values missing is more honest than making up numbers. I also confirm there are no impossible values (GDP per capita of zero or less). Finally I use the IQR method to find outliers. GDP per capita is very spread out, and the outliers are real values for small, very wealthy countries like Monaco and Liechtenstein. So instead of deleting them I add a new column that flags them for later analysis.

In [73]:
# Third data manipulation: identify bad data and outliers

# Bad data 1: some capitals are blank strings instead of missing values
# This will replace the NaN values with an empty string
blank_capital = countries_merged['capital'].str.strip() == ''
print(blank_capital.groupby(level='name').any().sum(), "countries with a blank capital")
countries_merged['capital'] = countries_merged['capital'].mask(blank_capital)  # blank -> NaN so it counts as missing

# Bad data 2: countries with no GDP value in any year add nothing to the analysis
# If this is the case we can probably just leave them out but I didn't see any
no_gdp = countries_merged['gdp-per-capita-usd'].isna().groupby(level='name').all()
print("Countries with no GDP data:", no_gdp[no_gdp].index.tolist())
countries_merged = countries_merged.drop(index=no_gdp[no_gdp].index, level='name')

# Bad data 3: GDP per capita can never be zero or negative
# A country can't exist without any income
print((countries_merged['gdp-per-capita-usd'] <= 0).sum(), "rows with GDP per capita <= 0")

# Outliers: flag values outside 1.5 * IQR. These are real values (e.g. Monaco), so I flag them instead of deleting them.
# I found a few outliers but these are actual countries usually they are very small reflecting a small GDP
gdp_col = countries_merged['gdp-per-capita-usd']
q1, q3 = gdp_col.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
countries_merged['gdp-outlier'] = (gdp_col < lower) | (gdp_col > upper)
print(f"Outlier bounds: {lower:,.0f} to {upper:,.0f} USD")
print(countries_merged['gdp-outlier'].sum(), "rows flagged as outliers")

# Countries with the most outlier years
countries_merged[countries_merged['gdp-outlier']].groupby(level='name').size().sort_values(ascending=False).head(10)

6 countries with a blank capital
Countries with no GDP data: ['British Virgin Islands', 'Gibraltar', "Korea, Dem. People's Rep."]
0 rows with GDP per capita <= 0
Outlier bounds: -31,025 to 58,298 USD
244 rows flagged as outliers


name
Norway            16
Luxembourg        16
Switzerland       16
Liechtenstein     15
Qatar             15
Monaco            15
Cayman Islands    15
Bermuda           15
Isle of Man       14
Denmark           12
dtype: int64

## Transformation 4: Find duplicates
I will check for duplicates in several ways: rows that are exact copies, a country appearing more than once in the same year, and a country code matching more than one country name (or one name matching more than one code). The World Bank API is a well maintained source, so I expect to find none, but checking confirms each row is one unique country-year. I also remove any duplicates so the step still works if a later pull from the API returns repeated records.

In [74]:
# Fourth data manipulation: find and remove duplicates

# Exact duplicate rows (reset_index so the name and year are part of the comparison)
print(countries_merged.reset_index().duplicated().sum(), "fully duplicated rows")

# Each country should appear only once per year
print(countries_merged.index.duplicated().sum(), "duplicate name/year index entries")

# Each country code should belong to exactly one country, and each country to exactly one code
flat = countries_merged.reset_index()
print((flat.groupby('alpha-3')['name'].nunique() > 1).sum(), "country codes used by more than one name")
print((flat.groupby('name')['alpha-3'].nunique() > 1).sum(), "names used by more than one country code")

# Keep only the first copy of any duplicate so the dataset stays clean if the API ever returns repeats
rows_before = len(countries_merged)
countries_merged = countries_merged[~countries_merged.index.duplicated(keep='first')]
print(rows_before - len(countries_merged), "rows removed")

0 fully duplicated rows
0 duplicate name/year index entries
0 country codes used by more than one name
0 names used by more than one country code
0 rows removed


## Transformation 5: Fix casing and inconsistent values
The World Bank country names do not follow one convention. Some put the descriptor after a comma ("Korea, Rep.", "Bahamas, The"), some abbreviate ("St. Lucia"), and the territory labels are inconsistent ("Puerto Rico (US)" vs "Virgin Islands (U.S.)"). I will add a `name-cleansed` column using the same rules as Milestone 2 so the names match across my datasets. I keep the original `name` in the index so no information is lost. Splitting on the comma would make "Congo, Rep." and "Congo, Dem. Rep." the same name, so I set those (and the two Koreas) by country code and then check that no two countries share a cleansed name. Finally I trim whitespace in the text columns, change "&" to "and" in the region names, and check that the category columns use consistent casing.

In [75]:
# Fifth data manipulation: fix inconsistent country names and text values
# I am trying to stay consistent with the data as the previous milestone so it will closely reflect

# The World Bank names mix several conventions: "Korea, Rep.", "Bahamas, The", "St. Lucia",
# "Puerto Rico (US)" vs "Virgin Islands (U.S.)". This cleans them the same way as name-cleansed in Milestone 2.
def cleanse_name(name):
    name = re.sub(r'\s*\(.*?\)', '', name)      # remove parentheticals like (US) or (French part)
    name = name.split(',')[0]                    # remove any text after a comma, same as Milestone 2
    name = re.sub(r'^St\. ', 'Saint ', name)     # spell out the "St." abbreviation
    name = re.sub(r' SAR$', '', name)            # "Hong Kong SAR" -> "Hong Kong"
    return name.strip()

countries_merged.insert(0, 'name-cleansed', [cleanse_name(n) for n in countries_merged.index.get_level_values('name')])

# Splitting on the comma makes some countries ambiguous, so fix those by their country code
manual_names = {'KOR': 'South Korea', 'PRK': 'North Korea',
                'COD': 'Democratic Republic of the Congo', 'COG': 'Republic of the Congo'}
for code, clean_name in manual_names.items():
    countries_merged.loc[countries_merged['alpha-3'] == code, 'name-cleansed'] = clean_name

# Remove accents and special characters, same as Milestone 2
countries_merged['name-cleansed'] = countries_merged['name-cleansed'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('ascii')

# Make sure the cleansing did not turn two different countries into the same name
names_per_clean = countries_merged.groupby('name-cleansed')['alpha-3'].nunique()
print((names_per_clean > 1).sum(), "cleansed names shared by more than one country")

# Text columns: trim stray whitespace and use "and" instead of "&" in the region names
for col in ['region', 'income-level', 'capital']:
    countries_merged[col] = countries_merged[col].str.strip()
countries_merged['region'] = countries_merged['region'].str.replace('&', 'and')

# Casing check: if lowercasing a column reduces its number of distinct values, the same value is cased differently
for col in ['region', 'income-level']:
    print(col, "- consistent casing:", countries_merged[col].nunique() == countries_merged[col].str.lower().nunique())

# Show only the names that changed
changed = countries_merged.reset_index()[['name', 'name-cleansed']].drop_duplicates()
changed[changed['name'] != changed['name-cleansed']]

0 cleansed names shared by more than one country
region - consistent casing: True
income-level - consistent casing: True


,name,name-cleansed
208,"Bahamas, The",Bahamas
688,"Congo, Dem. Rep.",Democratic Republic of the Congo
704,"Congo, Rep.",Republic of the Congo
912,"Egypt, Arab Rep.",Egypt
1120,"Gambia, The",Gambia
1344,"Hong Kong SAR, China",Hong Kong
1424,"Iran, Islamic Rep.",Iran
1616,"Korea, Rep.",South Korea
1824,"Macao SAR, China",Macao
2000,"Micronesia, Fed. Sts.",Micronesia


## Unique Country Names
A list of every country left in the cleaned dataset, using the cleansed names.

This is to just show that there are many countries of data included as in most of the head commands it just shows Afghanastan

In [76]:
# Unique country names in the final cleaned dataset
unique_names = sorted(countries_merged['name-cleansed'].unique())
print(len(unique_names), "unique countries\n")
print("\n".join(unique_names))

214 unique countries

Afghanistan
Albania
Algeria
American Samoa
Andorra
Angola
Antigua and Barbuda
Argentina
Armenia
Aruba
Australia
Austria
Azerbaijan
Bahamas
Bahrain
Bangladesh
Barbados
Belarus
Belgium
Belize
Benin
Bermuda
Bhutan
Bolivia
Bosnia and Herzegovina
Botswana
Brazil
Brunei Darussalam
Bulgaria
Burkina Faso
Burundi
Cabo Verde
Cambodia
Cameroon
Canada
Cayman Islands
Central African Republic
Chad
Channel Islands
Chile
China
Colombia
Comoros
Costa Rica
Cote d'Ivoire
Croatia
Cuba
Curacao
Cyprus
Czechia
Democratic Republic of the Congo
Denmark
Djibouti
Dominica
Dominican Republic
Ecuador
Egypt
El Salvador
Equatorial Guinea
Eritrea
Estonia
Eswatini
Ethiopia
Faroe Islands
Fiji
Finland
France
French Polynesia
Gabon
Gambia
Georgia
Germany
Ghana
Greece
Greenland
Grenada
Guam
Guatemala
Guinea
Guinea-Bissau
Guyana
Haiti
Honduras
Hong Kong
Hungary
Iceland
India
Indonesia
Iran
Iraq
Ireland
Isle of Man
Israel
Italy
Jamaica
Japan
Jordan
Kazakhstan
Kenya
Kiribati
Kosovo
Kuwait
Kyrgyz Republi